# To Find Out Why USI 10 ms Has Better Probability of NPRL

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import glob, os

## For Single Test Case

In [52]:
DATASET_ROOT = "/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Obj3_Retransmission_Datasets"
SCENARIO = "U_Dist_UAV_Interference_new/uav_scenario_1"
DELAY_THRESHOLD = 0.04

test_cases = [f.path for f in os.scandir(os.path.join(DATASET_ROOT, SCENARIO)) if f.is_dir()]

usi_switch_time_arr_list = []
usi_tw_rel_arr_list = []
for usi in ["10", "20", "66.7", "100"]:
    usi_test_cases = [x for x in test_cases if "_UAVSendingInterval-{}_".format(usi) in x ]
    switch_time_arr = []
    tw_rel_arr = []
    for test_case in usi_test_cases:
        runs = sorted(glob.glob("{}/Run-*_GCS-Tx.csv".format(test_case))) # Get the different runs for each scenario
        run_number = [run.split('/')[-1].split("_")[0].split("-")[-1] for run in runs]
        run_file_path = os.path.join(test_case, "Run-{}_{}.csv")
        for num in run_number:
            # Get the switch time
            switch_df = pd.read_csv(run_file_path.format(num, "Switch_Time"))
            if switch_df["Switch_Time"].min() < 1: # This is a case where no switching happened
                continue
            switch_node = switch_df.loc[switch_df['Switch_Time'].idxmin()]
            switch_time = float(switch_node["Switch_Time"])
            # Find out which node switched
            if ".GCS." in switch_node["Module"]:
                node = "GCS"
            elif ".gatewayNode." in switch_node["Module"]:
                node = "GW"
            elif ".adhocNode[0]." in switch_node["Module"]:
                node = "UAV-0"
            elif ".adhocNode[1]." in switch_node["Module"]:
                node = "UAV-1"
            elif ".adhocNode[2]." in switch_node["Module"]:
                node = "UAV-2"
            elif ".adhocNode[3]." in switch_node["Module"]:
                node = "UAV-3"
            elif ".adhocNode[4]." in switch_node["Module"]:
                node = "UAV-4"
            elif ".adhocNode[5]." in switch_node["Module"]:
                node = "UAV-5"
            elif ".adhocNode[6]." in switch_node["Module"]:
                node = "UAV-6"

            # Get the latest time window reliability that caused switching
            rx_df = pd.read_csv(run_file_path.format(num, "{}-Rx".format(node)))
            pd_df = pd.read_csv(run_file_path.format(num, "{}-PacketDrop".format(node)))
            rx_df_in_range = rx_df.loc[(rx_df["RxTime"] >= (switch_time-1)) & (rx_df["RxTime"] <= (switch_time))]
            rx_df_in_range["Delay"] = rx_df_in_range['RxTime'] - rx_df_in_range['TxTime']
            rel_df_in_range = rx_df_in_range.loc[(rx_df_in_range["Delay"] <= DELAY_THRESHOLD)]
            delay_df_in_range = rx_df_in_range.loc[(rx_df_in_range["Delay"] > DELAY_THRESHOLD)]
            pd_df_in_range = pd_df.loc[(pd_df["RxTime"] >= (switch_time-1)) & (pd_df["RxTime"] <= (switch_time))]
            time_win_reliability = len(rel_df_in_range)/(len(rel_df_in_range) + len(pd_df_in_range) + len(delay_df_in_range))

            switch_time_arr.append(switch_time)
            tw_rel_arr.append(time_win_reliability)
            
    usi_switch_time_arr_list.append(switch_time_arr)
    usi_tw_rel_arr_list.append(tw_rel_arr)

            




            

/tmp/ipykernel_1923204/4203455224.py:48: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rx_df_in_range["Delay"] = rx_df_in_range['RxTime'] - rx_df_in_range['TxTime']
/tmp/ipykernel_1923204/4203455224.py:48: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rx_df_in_range["Delay"] = rx_df_in_range['RxTime'] - rx_df_in_range['TxTime']
/tmp/ipykernel_1923204/4203455224.py:48: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value i

In [ ]:
print("Mean: {}, {}, {}, {}".format(np.mean(usi_tw_rel_arr_list[0]), np.mean(usi_tw_rel_arr_list[1]), np.mean(usi_tw_rel_arr_list[2]), np.mean(usi_tw_rel_arr_list[3])))
print("Max: {}, {}, {}, {}".format(np.max(usi_tw_rel_arr_list[0]), np.max(usi_tw_rel_arr_list[1]), np.max(usi_tw_rel_arr_list[2]), np.max(usi_tw_rel_arr_list[3])))
print("Min: {}, {}, {}, {}".format(np.min(usi_tw_rel_arr_list[0]), np.min(usi_tw_rel_arr_list[1]), np.min(usi_tw_rel_arr_list[2]), np.min(usi_tw_rel_arr_list[3])))

Mean: 0.12634626928819978, 0.2773461242804523, 0.7226641418001257, 0.7740751302839602
Max: 0.3557312252964427, 1.0, 1.0, 1.0
Mean: 0.009345794392523364, 0.0, 0.0, 0.0


In [56]:
print("Mean: {}, {}, {}, {}".format(np.mean(usi_switch_time_arr_list[0]), np.mean(usi_switch_time_arr_list[1]), np.mean(usi_switch_time_arr_list[2]), np.mean(usi_switch_time_arr_list[3])))
print("Max: {}, {}, {}, {}".format(np.max(usi_switch_time_arr_list[0]), np.max(usi_switch_time_arr_list[1]), np.max(usi_switch_time_arr_list[2]), np.max(usi_switch_time_arr_list[3])))
print("Min: {}, {}, {}, {}".format(np.min(usi_switch_time_arr_list[0]), np.min(usi_switch_time_arr_list[1]), np.min(usi_switch_time_arr_list[2]), np.min(usi_switch_time_arr_list[3])))

Mean: 1.0, 1.027183979974969, 1.3937227550130777, 1.9296566393193557
Max: 1.0, 3.4, 10.2, 18.7
Min: 1.0, 1.0, 1.0, 1.0
